## Utils — JSON, décorateurs et générateurs

Ce notebook contient des fonctions utilitaires pour :

- sauvegarder et charger les données JSON
- tracer l’exécution avec un décorateur `logger`
- vérifier l’existence du fichier JSON
- filtrer les examens/étudiants avec des générateurs

In [ ]:
# Imports de base pour JSON, fichiers et horodatage

import json
import os
from datetime import datetime

# Import des classes métier
from gestion_examens import *

## Décorateurs

Dans cette partie, j’ai implémenté deux décorateurs :

- `logger` : affiche le début et la fin de l’exécution d’une fonction ainsi que son résultat ou une éventuelle erreur.
- `verifier_json` : vérifie si le fichier JSON existe avant de charger les données.

> Chaque décorateur utilise la fonction interne **`wrapper`**.  
> Cette fonction permet d'intercepter l’appel de la fonction originale afin d’ajouter du code avant et après son exécution tout en passant les arguments nécessaires.

In [ ]:
def logger(fonction):

    # Fonction interne (wrapper) qui va remplacer la fonction originale
    def wrapper(*args, **kwargs):

        # Affiche la date et l'heure du début d'exécution de la fonction
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Debut de {fonction.__name__}")

        try:
            # Exécute la fonction originale avec ses arguments
            resultat = fonction(*args, **kwargs)

            # Vérifie si la fonction retourne une valeur ou non
            if resultat is None:
                # Message si la fonction s'est exécutée correctement sans retour
                print(f"[INFO] {fonction.__name__} exécutée avec succès.")
            else:
                # Affiche le résultat retourné par la fonction
                print(f"[INFO] Résultat de {fonction.__name__} : {resultat}")

            # Affiche la date et l'heure de fin d'exécution
            print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Fin de {fonction.__name__}")
            print("-" * 40)

            # Retourne le résultat de la fonction
            return resultat
        
        # Gestion des erreurs si une exception se produit
        except Exception as e:
            print(f"[ERREUR] La fonction {fonction.__name__} a échoué : {e}")
            print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Fin de {fonction.__name__}")
            print("-" * 40)

    # Retourne le wrapper qui remplace la fonction originale
    return wrapper


# Décorateur qui vérifie si le fichier JSON existe avant d'exécuter la fonction
def verifier_json(fonction):

    # Wrapper qui prend le nom du fichier JSON en paramètre
    def wrapper(fichier="data.json"):

        # Vérifie si le fichier existe dans le système
        if not os.path.exists(fichier):
            print("[ERREUR] Le fichier data.json n'existe pas.")

            # Retourne un nouvel objet CentreExamens si le fichier est absent
            return CentreExamens()

        # Si le fichier existe, la fonction originale est exécutée
        return fonction(fichier)

    return wrapper

## Sauvegarde des données

Cette fonction permet de **sauvegarder les données du centre d’examens dans un fichier JSON**.

Elle parcourt la liste des **étudiants** et des **examens** du centre et convertit chaque objet en dictionnaire à l’aide de la méthode `to_dict()`.  
Les données sont ensuite stockées dans un dictionnaire puis enregistrées dans le fichier `data.json`.

>La fonction est décorée avec `@logger`, ce qui permet d’afficher des informations sur l’exécution de la fonction (début, fin et éventuel résultat).

In [ ]:
@logger
def sauvegarder_donnees(centre, fichier="data.json"):
    data = {
        "etudiants": [],
        "examens": []
    }

    for e in centre.etudiants:
        data["etudiants"].append(e.to_dict())

    for ex in centre.examens:
        data["examens"].append(ex.to_dict())

    with open(fichier, "w") as f:
        json.dump(data, f, indent=4)


## Chargement des données

Cette fonction permet de **charger les données du centre d’examens depuis un fichier JSON**.

Elle lit le fichier `data.json`, puis recrée les objets **Etudiant** et **Examen** à partir des données enregistrées.  
Chaque objet est ensuite ajouté au **CentreExamens**.

>La fonction utilise le décorateur `@verifier_json` afin de vérifier que le fichier JSON existe avant de tenter de charger les données.  
>Une gestion des exceptions est également mise en place pour traiter les erreurs possibles (fichier inexistant, fichier JSON invalide, etc.).

In [ ]:
@verifier_json
def charger_donnees(fichier="data.json"):
    centre = CentreExamens()
    try:
        with open(fichier,"r") as f:
            data = json.load(f)
    
        # charger les etudiants et les examens
        for e in data["etudiants"]:
            etudiant = Etudiant(
                e["nom"],
                e["numero"],
                e["filiere"]
            )
            centre.ajouter_etudiant(etudiant)

        for ex in data["examens"]:
            examen = Examen( 
                ex["matiere"],
                ex["date"],
                ex["surveillant"]
            )
            centre.ajouter_examen(examen)
        
        print("Données chargées avec succès.")

    except FileNotFoundError:
        print("Fichier inexistant. Aucune donnée chargée.")

    except json.JSONDecodeError:
        print("Fichier JSON vide ou corrompu.")

    except Exception as e:
        print("Erreur lors du chargement :", e)

    return centre

## Générateurs de filtrage

Ces fonctions renvoient des éléments à la demande (`yield`) pour éviter de charger toute une liste filtrée en mémoire.

In [ ]:
# Retourne les examens correspondant à une date
def examens_par_date(examens, date):
    for ex in examens:
        if ex.date == date:
            yield ex


# Retourne les examens surveillés par une personne donnée
def examens_par_surveillant(examens, surveillant):
    for ex in examens:
        if ex.surveillant == surveillant:
            yield ex


# Retourne les étudiants d'une filière donnée
def etudiants_par_filiere(etudiants, filiere):
    for e in etudiants:
        if e.filiere == filiere:
            yield e